# Introduction

When working with Large Language Models (LLMs) we often tend to use datasets composed by natural language text written with the latin alphabet. This has proven effective but there are some subtelties hidden in such a choice. The english language has many heteronyms, as in words that are spelled the same but have different meaning and pronounciation. Also the phonetics of a word can help relate it to other words or concepts even if their spelling could be not much alike.

The International Phonetics Alphabet (IPA) provides an extended alphabet that precisely describes how words are pronounced. The usage of such an extended alphabet, instead of the latin alphabet, could help LLMs to improve there understanding of the dataset and consequently their accuracy in inference.

The project takes a well known dataset, the imdb movie reviews, and generates an analogous IPA dataset by using the [phonemizer](https://pypi.org/project/phonemizer/) library.

Then it uses the original and IPA datasets to train two models and then compares the results.

# Setup

In [ ]:
!pip install transformers[torch] phonemizer torch pandas
!pip install accelerate -U
!apt-get install -y espeak

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
espeak is already the newest version (1.48.15+dfsg-3).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Imports and Dataset Load

In [ ]:
from urllib.request import urlretrieve

def download(file, url):
    if not os.path.isfile(file):
        urlretrieve(url, file)

def strip_tags(text):
    return text.replace("<br />", "\n")

In [ ]:
import os
import pandas as pd


download("imdb-train.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-train.csv.gz")
train_set = pd.read_csv("imdb-train.csv.gz", sep="\t", names=["label", "text"])
train_set["text"] = train_set["text"].apply(strip_tags)
download("imdb-test.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-test.csv.gz")
test_set = pd.read_csv("imdb-test.csv.gz", sep="\t", names=["label", "text"])
test_set["text"] = test_set["text"].apply(strip_tags)


### Dataset preparation

In [ ]:
from sklearn import preprocessing
from transformers import AutoModelForSequenceClassification, AutoTokenizer, AutoConfig
from sklearn.model_selection import train_test_split
import torch

class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

def encode_labels(labels):
    le = preprocessing.LabelEncoder()
    return le.fit_transform(labels)

def prepare_dataset(tokenizer, dataset):
    labels = encode_labels(dataset["label"])
    texts = dataset["text"]
    if type(texts) != list:
        texts = texts.tolist()

    encodings = tokenizer(texts, truncation=True, padding=True)
    dataset = IMDbDataset(encodings, labels)
    return dataset

def prepare_datasets(tokenizer_params, full_train_set, test_set):
    train_set = {}
    val_set = {}
    tokenizer = AutoTokenizer.from_pretrained(**tokenizer_params)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token # manually set pad_token for gpt2 model
    train_set["text"], val_set["text"], train_set["label"], val_set["label"] = train_test_split(full_train_set["text"].tolist(), full_train_set["label"].tolist(), test_size=.1)
    train_dataset = prepare_dataset(tokenizer, train_set)
    val_dataset = prepare_dataset(tokenizer, val_set)
    test_dataset = prepare_dataset(tokenizer, test_set)
    return train_dataset, val_dataset, test_dataset, tokenizer.pad_token_id


# Phonemize Text (Convert to IPA)

In [ ]:
from urllib.request import urlretrieve
from phonemizer import phonemize
import pandas as pd
import re

def phonemize_batch(batch):
    # split each text into parts (keeping track of structure)
    structured_parts = []
    all_text_parts = []

    for text in batch:
        parts = re.split(r'(<br\s*/?>)', text, flags=re.IGNORECASE)
        current_structure = []
        for part in parts:
            if re.match(r'<br\s*/?>', part, flags=re.IGNORECASE) or part.strip() == "":
                # keep HTML and empty parts as-is
                current_structure.append((part, False))
            else:
                # mark normal text part for phonemization
                current_structure.append((len(all_text_parts), True))
                all_text_parts.append(part)
        structured_parts.append(current_structure)

    # phonemize all text parts at once
    phonemized_all = phonemize(
        all_text_parts,
        language='en-us',
        backend='espeak',
        strip=False,
        preserve_punctuation=True
    )

    # reconstruct the original texts efficiently
    result = []
    for structure in structured_parts:
        reconstructed = []
        for part, is_text in structure:
            if is_text:
                reconstructed.append(phonemized_all[part])
            else:
                reconstructed.append(part)
        result.append("".join(reconstructed))
    return result

def save_phonemized_data(ipa_data, filename):
    ipa_data.to_csv(filename, index=False, encoding='utf-8')

def generate_ipa_dataset():
    download("imdb-train.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-train.csv.gz")
    train_set = pd.read_csv("imdb-train.csv.gz", sep="\t", names=["label", "text"])
    download("imdb-test.csv.gz", "https://github.com/datascienceunibo/bbs-dl-lab-2019/raw/master/imdb-test.csv.gz")
    test_set = pd.read_csv("imdb-test.csv.gz", sep="\t", names=["label", "text"])

    # Phonemize dataset
    train_set["text"] = phonemize_batch(train_set["text"])
    save_phonemized_data(train_set, "ipa_dataset/ipa_train.csv")
    test_set["text"] = phonemize_batch(test_set["text"])
    save_phonemized_data(test_set, "ipa_dataset/ipa_test.csv")

Since the generation of the IPA dataset is very time consuming, the dataset has already been generated and made available through github.

In [ ]:
import requests
import os
import json
import gc
import pandas as pd

def load_phonemized_data(filename):
    return pd.read_csv(filename, encoding='utf-8')

train_url = "https://raw.githubusercontent.com/Oldranda1414/ipa_bert_test/refs/heads/main/imdb_ipa_dataset/ipa_train.csv"
test_url = "https://raw.githubusercontent.com/Oldranda1414/ipa_bert_test/refs/heads/main/imdb_ipa_dataset/ipa_test.csv"

for url in [train_url, test_url]:
    filename = os.path.basename(url)
    download(filename, url)

ipa_train_set = load_phonemized_data("ipa_train.csv")
ipa_test_set = load_phonemized_data("ipa_test.csv")

ipa_train_set["text"] = ipa_train_set["text"].apply(strip_tags)
ipa_test_set["text"] = ipa_test_set["text"].apply(strip_tags)


The Tokenizer for the IPA model expects input to be space separated characters with WORD_BOUNDARY between words. The following function preprocesses the generated IPA dataset further to allign it with the ipa tokenizer's expectations:

In [ ]:
import re

def prepare_ipa_text(text):
    # Split words by space (if words are already separated)
    words = text.strip().split()

    processed_words = []
    for word in words:
        # Split into individual phonemes (characters)
        phonemes = list(word)
        # Join with space and insert WORD_BOUNDARY between words
        processed_words.append(" ".join(phonemes))

    # Join all words with WORD_BOUNDARY markers
    return " WORD_BOUNDARY ".join(processed_words)

# Training Function

In [ ]:
from transformers import Trainer, TrainingArguments, AutoModelForSequenceClassification
import torch

from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

def train_model(model_params, train_dataset, val_dataset, test_dataset, pad_token_id=None):
    model = AutoModelForSequenceClassification.from_pretrained(**model_params, num_labels=2)
    if pad_token_id is not None:
        model.config.pad_token_id = pad_token_id

    args = TrainingArguments(
        output_dir=f"./results-{model_params["pretrained_model_name_or_path"]}",
        num_train_epochs=1,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=64,
        warmup_steps=500,
        weight_decay=0.01,
        logging_dir='./logs',
        logging_steps=10,
        do_eval=True,
        eval_steps=200,
        report_to="none",
        optim="adamw_torch", # To solved fused=True error on TPU
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics
    )

    trainer.train()
    metrics = trainer.evaluate(test_dataset)
    return metrics


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:82: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


# Train models

In [ ]:
import gc
model_name = "distilbert-base-uncased"
train_dataset, val_dataset, test_dataset, _ = prepare_datasets({"pretrained_model_name_or_path":model_name}, train_set, test_set)
print("Training baseline model...")
metrics_base = train_model({"pretrained_model_name_or_path":model_name}, train_dataset, val_dataset, test_dataset)
print("Baseline:", metrics_base)


Training baseline model...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:666: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
10,0.690000
20,0.700200
30,0.692100
40,0.683600
50,0.682900
60,0.683800
70,0.679200
80,0.641500
90,0.594400
100,0.505400


KeyboardInterrupt: 

In [ ]:
import gc
ipa_model_name = {"pretrained_model_name_or_path":'phonemetransformers/ipa-childes-models-tiny', "subfolder":'EnglishUK'}
ipa_tokenizer_name = {"pretrained_model_name_or_path":'phonemetransformers/ipa-childes-tokenizers', "subfolder":'EnglishUK'}
for data_set in [ipa_train_set, ipa_test_set]:
    data_set["text"] = data_set["text"].apply(prepare_ipa_text)
#ipa_train_dataset, ipa_val_dataset, ipa_test_dataset, pad_token_id = prepare_datasets(ipa_tokenizer_name, ipa_train_set, ipa_test_set)
#print("Training ipa model...")
#metrics_ipa = train_model(ipa_model_name, ipa_train_dataset, ipa_val_dataset, ipa_test_dataset, pad_token_id)
#print("Baseline:", metrics_ipa)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(**ipa_tokenizer_name)

sample_texts = ipa_train_set["text"].apply(prepare_ipa_text)[:5].to_list()
enc = tokenizer(sample_texts, padding=True, truncation=True, return_tensors="pt")

print("Tokens:", enc["input_ids"])
print("Decoded back:", [tokenizer.decode(ids) for ids in enc["input_ids"]])

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Tokens: tensor([[ 3, 39,  2,  ...,  0,  2,  0],
        [ 3, 26,  2,  ...,  1,  1,  1],
        [ 3, 12,  2,  ...,  1,  1,  1],
        [ 3, 29,  2,  ...,  1,  1,  1],
        [ 3, 39,  2,  ...,  1,  1,  1]])
Decoded back: ['UTT_BOUNDARY b WORD_BOUNDARY ɹ WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY m WORD_BOUNDARY w WORD_BOUNDARY UNK WORD_BOUNDARY l WORD_BOUNDARY UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK WORD_BOUNDARY h WORD_BOUNDARY UNK WORD_BOUNDARY ɪ WORD_BOUNDARY UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK WORD_BOUNDARY ɪ WORD_BOUNDARY z WORD_BOUNDARY UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK WORD_BOUNDARY ɐ WORD_BOUNDARY UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY ɹ WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY n WORD_BOUNDARY UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK UNK WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY UNK WORD_BOUNDARY m WORD

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(**ipa_tokenizer_name)
print(tokenizer.decode(tokenizer.encode("a big bird")))

a ɪ ɐ ʃ WORD_BOUNDARY b ɪ ɾ d
UTT_BOUNDARY UNK ɪ ɐ ʃ WORD_BOUNDARY b ɪ UNK d


In [ ]:
samples = ["aɪ ɐ ʃ", "aɪɐʃ"]  # three variants

for s in samples:
    enc = tokenizer(s)
    print(f"Input: {s}")
    print("Tokens:", tokenizer.convert_ids_to_tokens(enc["input_ids"]))
    print("Decoded:", tokenizer.decode(enc["input_ids"]))
    print("---")

Input: aɪ ɐ ʃ
Tokens: ['UTT_BOUNDARY', 'aɪ', 'ɐ', 'ʃ']
Decoded: UTT_BOUNDARY aɪ ɐ ʃ
---
Input: aɪɐʃ
Tokens: ['UTT_BOUNDARY', 'UNK']
Decoded: UTT_BOUNDARY UNK
---


# Compare Results

In [ ]:
print("✅ Baseline Accuracy:", metrics_base["eval_accuracy"])
print("✅ IPA Accuracy:", metrics_ipa["eval_accuracy"])

# Conclusions

